# MySQL

Leemos el archivo csv y lo almacenamos en un dataframe de pandas

In [ ]:
import pandas as pd

df = pd.read_csv('spotify-clean.csv')
df.sample(5)

,Unnamed: 0.1,Unnamed: 0,track_id,artists,album_name,track_name,popularity,duration_ms
12468,13304,13304,6JKxHuasmtx1gpUcysUOwc,Mark Farina;Homero Espinosa,We Got It,We Got It,10,228186
22631,23965,23965,5nnfso5MO99iNnxvNBR5Bs,Acnatro;Amorf,Heaven,Heaven,31,196806
2876,2878,2878,5n40SAKMk89wdWyhTfWzx4,Gondwana,RAS Portraits,Antonia,45,248213
23960,25366,25366,4EhkMUdkAVAsfsjbWU6PPr,The Bangles,Different Light,Manic Monday,46,184173
9640,10374,10374,5xDQOEHXtk014zxxXtdHfM,Ed Solo;Skool Of Thought;Bukue One;Pimpernal J...,Random Acts of Kindness,Sometimes,15,267352


Antes de conectarnos a nuestra BBDD de MySQL, instalaremos en nuestro environment: `pip install mysql-connector-python`.

Ahora, nos conectaremos a nuestra BBDD 'pipe' de MySQL y verificamos qué databases están creadas

In [ ]:
import mysql.connector

mydb = mysql.connector.connect(
  host="localhost",
  user="root",
  password="changeme",
  database="pipe"
)

mycursor = mydb.cursor(buffered=True)

mycursor.execute("SHOW TABLES")

for x in mycursor:
  print(x)


('track',)


In [ ]:
mycursor.execute("DROP TABLE track")

A continuación nos conectaremos a la BBDD 'pipe' y crearemos en ella la tabla 'spotify_tracks'

In [ ]:
mycursor.execute("CREATE TABLE IF NOT EXISTS track (" \
"track_id varchar(250), " \
"track_name varchar(250), " \
"artists varchar(1000)," \
"album_name varchar(5120)," \
"popularity int," \
"duration_ms bigint," \
"PRIMARY KEY (track_id)" \
");")

#Show table structure
mycursor.execute("DESCRIBE track")

for x in mycursor:
  print(x)

('track_id', 'varchar(250)', 'NO', 'PRI', None, '')
('track_name', 'varchar(250)', 'YES', '', None, '')
('artists', 'varchar(1000)', 'YES', '', None, '')
('album_name', 'varchar(5120)', 'YES', '', None, '')
('popularity', 'int', 'YES', '', None, '')
('duration_ms', 'bigint', 'YES', '', None, '')


Comprobamos que la creación de la tabla `tracks` fue correcta. Procedemos a la inserción de los datos de nuestro dataset en la BBDD de MySQL.

Para ello, crearemos un bucle que nos inserte cada dato de nuestro dataframe en la tabla de mysql

In [ ]:
for i in range(len(df)):
    sql = "INSERT INTO track (track_id, track_name, artists, album_name, popularity, duration_ms) VALUES (%s, %s, %s, %s, %s, %s)"
    val = (
        df.iloc[i]['track_id'],
        df.iloc[i]['track_name'],
        df.iloc[i]['artists'],
        df.iloc[i]['album_name'],
        int(df.iloc[i]['popularity']),    # convertir a tipo int nativo
        int(df.iloc[i]['duration_ms'])    # convertir a tipo int nativo
    )
    mycursor.execute(sql, val)
    
mydb.commit()